In [ ]:
!pip install transformers -q

import os, pickle, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (Input, Dense, Dropout, BatchNormalization,
                                      Concatenate, GlobalAveragePooling2D, Lambda)
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing import image as keras_image
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import torch
from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ TF:", tf.version)
print("🖥️  Device:", DEVICE)

In [ ]:
!pip install transformers -q

import os, pickle, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (Input, Dense, Dropout, BatchNormalization,
                                      Concatenate, GlobalAveragePooling2D, Lambda)
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing import image as keras_image
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion__matrix, accuracy_score

import torch
from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ TF:", tf.version)
print("🖥️  Device:", DEVICE)


In [ ]:
IMG_SIZE      = (224, 224)
BERT_DIM      = 768
EMBED_DIM     = 256

img_input  = Input(shape=(*IMG_SIZE, 3), name="image_input")
resnet_base= ResNet50(weights="imagenet", include_top=False,
                       input_tensor=img_input)
resnet_base.trainable = False   # frozen

img_features = GlobalAveragePooling2D(name="img_gap")(resnet_base.output)
img_features = Dense(EMBED_DIM, activation="relu",  name="img_proj")(img_features)
img_features = BatchNormalization(name="img_bn")(img_features)
img_features = Dropout(0.3, name="img_drop")(img_features)

text_input   = Input(shape=(BERT_DIM,), name="text_input")
txt_features = Dense(EMBED_DIM, activation="relu",  name="txt_proj")(text_input)
txt_features = BatchNormalization(name="txt_bn")(txt_features)
txt_features = Dropout(0.3, name="txt_drop")(txt_features)

fused = Concatenate(name="fusion")([img_features, txt_features])
fused = Dense(512, activation="relu",  name="fused_dense1")(fused)
fused = Dropout(0.4,                   name="fused_drop1")(fused)
fused = Dense(256, activation="relu",  name="fused_dense2")(fused)
fused = Dropout(0.3,                   name="fused_drop2")(fused)

drug_name_out  = Dense(NUM_CLASSES, activation="softmax", name="drug_name")(fused)
risk_out       = Dense(3,           activation="softmax", name="risk_level")(fused)

fusion_model = Model(
    inputs =[img_input, text_input],
    outputs=[drug_name_out, risk_out],
    name   ="FusionModel"
)

fusion_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss={"drug_name":"categorical_crossentropy",
          "risk_level":"categorical_crossentropy"},
    loss_weights={"drug_name": 1.0, "risk_level": 0.5},
    metrics={"drug_name":"accuracy", "risk_level":"accuracy"}
)

fusion_model.summary()

In [ ]:
RISK_MAP = {
    "aspirin"      : 1,
    "ibuprofen"    : 1,
    "paracetamol"  : 0,
    "amoxicillin"  : 1,
    "azithromycin" : 1,
    "cetirizine"   : 0,
    "diclofenac"   : 1,
    "metformin"    : 1,
    "omeprazole"   : 0,
    "atorvastatin" : 1,
    "insulin"      : 2,
    "prednisone"   : 2,
    "salbutamol"   : 1,
}

DATA_DIR = "/content/drug_data/Drug Vision/Data Combined"

records = []
for cls_folder in sorted(os.listdir(DATA_DIR)):
    cls_path = os.path.join(DATA_DIR, cls_folder)
    if not os.path.isdir(cls_path): continue
    matched_drug = None
    for drug in TEXT_CLASSES:
        if drug.lower() in cls_folder.lower():
            matched_drug = drug; break
    if matched_drug is None:
        matched_drug = random.choice(TEXT_CLASSES)

    for img_file in os.listdir(cls_path)[:150]:
        records.append({
            "img_path"  : os.path.join(cls_path, img_file),
            "drug"      : matched_drug,
            "risk_level": RISK_MAP.get(matched_drug, 1)
        })

fusion_df = pd.DataFrame(records).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"✅ Fusion dataset: {len(fusion_df)} samples")
print(fusion_df["drug"].value_counts())

In [ ]:
idx = np.arange(len(fusion_df))
idx_tr, idx_tmp = train_test_split(idx, test_size=0.30, random_state=42)
idx_val, idx_te = train_test_split(idx_tmp, test_size=0.50, random_state=42)

def split(arr, idx): return arr[idx]

X_img_tr,  X_img_val,  X_img_te  = split(X_img,  idx_tr), split(X_img,  idx_val), split(X_img,  idx_te)
X_txt_tr,  X_txt_val,  X_txt_te  = split(X_text, idx_tr), split(X_text, idx_val), split(X_text, idx_te)
y_drug_tr, y_drug_val, y_drug_te = split(y_drug, idx_tr), split(y_drug, idx_val), split(y_drug, idx_te)
y_risk_tr, y_risk_val, y_risk_te = split(y_risk, idx_tr), split(y_risk, idx_val), split(y_risk, idx_te)

print(f"Train: {len(idx_tr)} | Val: {len(idx_val)} | Test: {len(idx_te)}")

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_drug_name_accuracy", patience=5,
                  restore_best_weights=True, verbose=1, mode='max'),
    ModelCheckpoint("best_fusion.h5", monitor="val_drug_name_accuracy",
                    save_best_only=True, verbose=1, mode='max'),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=4,
                      min_lr=1e-8, verbose=1)
]

history = fusion_model.fit(
    x=[X_img_tr, X_txt_tr],
    y={"drug_name": y_drug_tr, "risk_level": y_risk_tr},
    validation_data=(
        [X_img_val, X_txt_val],
        {"drug_name": y_drug_val, "risk_level": y_risk_val}
    ),
    epochs=5,
    batch_size=32,
    callbacks=callbacks
)
print("\n✅ Fusion model training complete!")